# 16S Analyses of the Longitudinal Acne Study
## Alpha Diversity Plots

Date last updated: 1/7/25

Notebook authors: Yang Chen, Britta De Pessemier, and Tyler Myers

This notebook plots the following:

- 16S V1-V3 and V4 Shannon alpha diversity plots at ASV level from rarefied abundance tables
- 16S V1-V3 and V4 Faith PD alpha diversity plots at ASV level from rarefied abundance tables

In [146]:
# Import Python packages
import re
import pandas as pd
import numpy as np
import biom
from biom import Table
from biom.util import biom_open
from biom import load_table
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
import os
from matplotlib.colors import ListedColormap
from matplotlib.colors import to_rgba
from skbio.diversity import alpha_diversity
from skbio.diversity.alpha import shannon
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from skbio import TreeNode
from skbio.diversity.alpha import faith_pd
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import spearmanr

In [147]:
# Load the metadata
metadata_path = '../Metadata/metadata_final_22102024_with_age.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group,severity_level,severity_group,subject_ID_x_group,age
0,LAMI.RD001.D14.C1,C1,not applicable,Non-lesional,skin,Day 14,not applicable,14,1,not applicable,...,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy,18
1,LAMI.RD001.D28.C1,C1,not applicable,Non-lesional,skin,Day 28,not applicable,28,1,not applicable,...,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy,18
2,LAMI.RD001.D0.C2,C2,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_Healthy,18
3,LAMI.RD001.D28.C2,C2,not applicable,Non-lesional,skin,Day 28,not applicable,28,1,not applicable,...,RD001,healthy,PP_1,PP_1C2,Non-lesional_C2,Healthy,absent,absent Healthy,PP_1_Healthy,18
4,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy,absent,absent Healthy,PP_1_Healthy,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD319.D16.C2,C2,not applicable,Lesional,skin,Day 16,not applicable,16,319,not applicable,...,RD319,acne,PP_319,PP_319C2,Lesional_C2,Acne_L,high,high Acne_L,PP_319_Acne_L,20
262,LAMI.RD319.D9.C1,C1,not applicable,Lesional,skin,Day 9,not applicable,9,319,not applicable,...,RD319,acne,PP_319,PP_319C1,Lesional_C1,Acne_L,moderate,moderate Acne_L,PP_319_Acne_L,20
263,LAMI.RD319.D2.C2,C2,not applicable,Lesional,skin,Day 2,not applicable,2,319,not applicable,...,RD319,acne,PP_319,PP_319C2,Lesional_C2,Acne_L,high,high Acne_L,PP_319_Acne_L,20
264,LAMI.RD319.D28.C2,C2,72,Lesional,skin,Day 28,13,28,319,14,...,RD319,acne,PP_319,PP_319C2,Lesional_C2,Acne_L,moderate,moderate Acne_L,PP_319_Acne_L,20


In [148]:
# Define paths to tables
biom_paths = {
    'V1-V3': '../Data/16S/Tables/intersection_samples_V1V3_V4/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_16S-aligned.biom',
    'V4': '../Data/16S/Tables/intersection_samples_V1V3_V4/174951_feature-table_16S_V4_rare-3769_sampleIDfixed_16S-aligned.biom'
}

In [149]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum
def load_biom_table(biom_path, metadata_path):
    # Load metadata as a DataFrame from the file path
    metadata = pd.read_csv(metadata_path, sep='\t')

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)
    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])
    
    return df

In [150]:
# Use sample samples from V1-V3 and V4 for alpha diversity analyses
def get_clean_sample_ids(biom_path):
    table = load_table(biom_path)
    sample_ids = pd.Series(list(table.ids(axis='sample')))

    sample_ids = (
        sample_ids
        .astype(str)
        .str.replace(r'^.*?(?=LAMI)', '', regex=True)
    )

    return set(sample_ids)

In [151]:
# Subset tables for paired-sample consistency
biom_v1v3 = biom_paths['V1-V3']
biom_v4   = biom_paths['V4']

samples_v1v3 = get_clean_sample_ids(biom_v1v3)
samples_v4   = get_clean_sample_ids(biom_v4)

shared_samples = samples_v1v3.intersection(samples_v4)

print(f"Shared samples between V1–V3 and V4: {len(shared_samples)}")

Shared samples between V1–V3 and V4: 184


In [152]:
# Load metadata
metadata = pd.read_csv(metadata_path, sep='\t')

# Ensure SampleID is string
metadata['SampleID'] = metadata['SampleID'].astype(str)

# Keep only samples shared between V1–V3 and V4
md_shared = metadata[metadata['SampleID'].isin(shared_samples)]

# Count per group
group_counts = (
    md_shared
    .groupby('group')
    .size()
    .reset_index(name='n')
)

print(group_counts)

     group    n
0   Acne_L  123
1  Acne_NL   42
2  Healthy   19


## Shannon Alpha Diversity

In [153]:
def calculate_shannon_alpha_diversity_and_plot(
    outdir,
    biom_path,
    metadata_path,
    group_col='group',
    subject_col='subject_ID',
    covariates=('age',),            # NEW: tuple of column names to adjust for
    title_suffix='',
    keep_samples=None,
    save_biom_path=None
):
    """
    Compute Shannon alpha diversity from a BIOM table, fit a mixed-effects model
    adjusting for the given covariates, compute pairwise group contrasts, and plot.
    """

    def mixedlm_pairwise_pvalues(df, response, group_col, subject_col, covariates):
        # Build formula: Shannon ~ group [+ cov1 + cov2 ...]
        rhs = group_col
        if covariates:
            rhs = rhs + " + " + " + ".join(covariates)
        formula = f"{response} ~ {rhs}"

        model = smf.mixedlm(formula, data=df, groups=df[subject_col])
        result = model.fit()

        params = result.params
        cov = result.cov_params()
        param_names = params.index.tolist()

        def make_contrast(terms):
            # zeros for every param, including the covariates — so group contrasts
            # are automatically "adjusted for age" without us doing anything extra
            c = np.zeros(len(params))
            for term, weight in terms.items():
                c[param_names.index(term)] = weight
            return c

        contrasts = {
            "Healthy vs Acne_NL": make_contrast({f"{group_col}[T.Acne_NL]": 1}),
            "Healthy vs Acne_L":  make_contrast({f"{group_col}[T.Acne_L]": 1}),
            "Acne_NL vs Acne_L":  make_contrast({
                f"{group_col}[T.Acne_L]": 1,
                f"{group_col}[T.Acne_NL]": -1
            })
        }

        rows = []
        for name, c in contrasts.items():
            est = np.dot(c, params.values)
            se = np.sqrt(np.dot(c, np.dot(cov.values, c)))
            z  = est / se
            p  = 2 * (1 - stats.norm.cdf(abs(z)))
            rows.append({"comparison": name, "estimate": est, "z": z, "p": p})

        return pd.DataFrame(rows), result

    table = load_table(biom_path)
    metadata = pd.read_csv(metadata_path, sep="\t")

    if keep_samples is not None:
        keep_samples = set(keep_samples)
        samples_in_table = {
            sid for sid in table.ids(axis="sample")
            if re.sub(r"^.*?(?=LAMI)", "", str(sid)) in keep_samples
        }
        table = table.filter(samples_in_table, axis="sample", inplace=False)

    if save_biom_path is not None:
        os.makedirs(os.path.dirname(save_biom_path), exist_ok=True)
        with h5py.File(save_biom_path, "w") as f:
            table.to_hdf5(f, "alpha-diversity-input")

    shannon_df = pd.DataFrame({
        "sample_id": list(table.ids(axis="sample")),
        "Shannon": [shannon(table.data(sid, axis="sample"))
                    for sid in table.ids(axis="sample")]
    })

    shannon_df["sample_id"] = (
        shannon_df["sample_id"].astype(str)
        .str.replace(r"^.*?(?=LAMI)", "", regex=True)
    )

    shannon_df.to_csv(f'../Analyses/16S/diversity/Shannon_{key}.csv')

    metadata["SampleID"] = metadata["SampleID"].astype(str)
    metadata = metadata.merge(
        shannon_df, left_on="SampleID", right_on="sample_id", how="inner"
    )

    # Coerce covariates to numeric (TSV loads everything as string/object)
    covariates = list(covariates) if covariates else []
    for cov_col in covariates:
        if cov_col in metadata.columns:
            metadata[cov_col] = pd.to_numeric(metadata[cov_col], errors="coerce")
        else:
            raise KeyError(f"Covariate '{cov_col}' not found in metadata columns.")

    # Drop rows missing any model variable (now includes covariates)
    required = [group_col, subject_col, "Shannon"] + covariates
    metadata = metadata.dropna(subset=required)

    metadata[group_col] = pd.Categorical(
        metadata[group_col],
        categories=["Healthy", "Acne_NL", "Acne_L"],
        ordered=False
    )

    pairwise_df, lmm_result = mixedlm_pairwise_pvalues(
        metadata, "Shannon", group_col, subject_col, covariates
    )

    print(f"\n[INFO] MixedLM results ({title_suffix}) — adjusted for: {covariates or 'none'}")
    print(lmm_result.summary())
    print(pairwise_df)


    box_palette = {
        "Healthy": "#3333B3",
        "Acne_NL": "#5cbccb",
        "Acne_L": "#f16c52"
    }

    darker_palette = {
        "Healthy": "#23238f",
        "Acne_NL": "#44a7b6"
    }

    severity_palette = {
        "low": "#f3b7a6",
        "moderate": "#e07b63",
        "high": "#b7372a"
    }

    fig, ax = plt.subplots(figsize=(4, 6))

    sns.boxplot(
        data=metadata,
        x=group_col,
        y="Shannon",
        order=["Healthy", "Acne_NL", "Acne_L"],
        palette=box_palette,
        showcaps=True,
        showfliers=False,
        boxprops=dict(edgecolor="black", linewidth=1.2),
        whiskerprops=dict(color="black", linewidth=1.2),
        medianprops=dict(color="black", linewidth=1.5),
        ax=ax
    )

    non_lesional = metadata[metadata[group_col] != "Acne_L"]

    sns.stripplot(
        data=non_lesional,
        x=group_col,
        y="Shannon",
        order=["Healthy", "Acne_NL", "Acne_L"],
        hue=group_col,
        hue_order=["Healthy", "Acne_NL"],
        palette=darker_palette,
        jitter=True,
        dodge=False,
        size=4,
        alpha=0.9,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
        zorder=2
    )

    ax.legend_.remove()

    if "severity_level" in metadata.columns:
        acne_l = metadata[metadata[group_col] == "Acne_L"]

        sns.stripplot(
            data=acne_l,
            x=group_col,
            y="Shannon",
            order=["Healthy", "Acne_NL", "Acne_L"],
            hue="severity_level",
            hue_order=["low", "moderate", "high"],
            palette=severity_palette,
            jitter=True,
            dodge=False,
            size=4,
            alpha=0.9,
            edgecolor="black",
            linewidth=0.4,
            ax=ax,
            zorder=3
        )

        handles, labels = ax.get_legend_handles_labels()

        label_map = {
            "low": "Low (1–2)",
            "moderate": "Mod (3–4)",
            "high": "High (5–6)"
        }

        severity_handles = []
        severity_labels = []

        for h, l in zip(handles, labels):
            if l in label_map:
                severity_handles.append(h)
                severity_labels.append(label_map[l])

        for h in severity_handles:
            h.set_sizes([16])
            h.set_edgecolor("black")
            h.set_linewidth(0.6)

        if title_suffix == "V1-V3":
            ax.legend(
                severity_handles,
                severity_labels,
                title="Severity",
                fontsize=8,
                title_fontsize=9,
                frameon=True,
                loc="upper right",
                bbox_to_anchor=(1.0, 0.85)
            )
        elif title_suffix == "V4":
            ax.legend(
                severity_handles,
                severity_labels,
                title="Severity",
                fontsize=8,
                title_fontsize=9,
                frameon=True,
                loc="lower left"
            )

    def add_pvalue(ax, x1, x2, y, p):
        stars = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
        label = f"{stars}  p={p:.2g}".strip()
        ax.plot([x1, x1, x2, x2], [y, y + 0.05, y + 0.05, y], lw=1.2, c="black")
        ax.text((x1 + x2) / 2, y + 0.06, label,
                ha="center", va="bottom", fontsize=8)

    ymax = metadata["Shannon"].max()
    step = ymax * 0.12
    current_y = ymax + step

    comparison_map = {
        "Healthy vs Acne_NL": ("Healthy", "Acne_NL"),
        "Healthy vs Acne_L":  ("Healthy", "Acne_L"),
        "Acne_NL vs Acne_L":  ("Acne_NL", "Acne_L")
    }

    for _, row in pairwise_df.iterrows():
        g1, g2 = comparison_map[row["comparison"]]
        x1 = ["Healthy", "Acne_NL", "Acne_L"].index(g1)
        x2 = ["Healthy", "Acne_NL", "Acne_L"].index(g2)
        add_pvalue(ax, x1, x2, current_y, row["p"])
        current_y += step * 0.8

    group_counts = metadata[group_col].value_counts().to_dict()

    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels([
        f"H\n(n={group_counts.get('Healthy', 0)})",
        f"ANL\n(n={group_counts.get('Acne_NL', 0)})",
        f"AL\n(n={group_counts.get('Acne_L', 0)})"
    ])

    ax.set_title(f"16S rRNA ({title_suffix}) Alpha Diversity")
    ax.set_xlabel("")
    ax.set_ylabel("Shannon Index")

    sns.despine()
    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(
        outdir,
        f"Shannon_alpha_diversity_{title_suffix.replace(' ', '_')}.png"
    )
    plt.savefig(outpath, dpi=600, bbox_inches="tight")
    plt.close(fig)

    # Plot diversity correlation with age
    # ---- Aggregate to subject-level ----
    subject_df = (
        metadata
        .groupby(subject_col)
        .agg({
            "Shannon": "mean",
            "age": "first"   # assumes age is constant per subject
        })
        .reset_index()
    )

    # ---- Spearman correlation (subject-level) ----
    rho, p_spear = spearmanr(subject_df["Shannon"], subject_df["age"])

    print(f"\n[INFO] Subject-level Shannon vs Age correlation:")
    print(f"Spearman ρ = {rho:.2f}\np = {p_spear:.4f}")

    # ---- Correlation plot (subject-level) ----
    fig_corr, ax_corr = plt.subplots(figsize=(4, 4))

    sns.regplot(
        data=subject_df,
        x="age",
        y="Shannon",
        scatter_kws=dict(s=50, edgecolor="black", linewidth=0.6),
        line_kws=dict(color="black", linewidth=1),
        ax=ax_corr
    )

    ax_corr.set_xlabel("Age")
    ax_corr.set_ylabel("Mean Shannon per subject")
    ax_corr.set_title(f"Subject-level Shannon vs Age ({title_suffix})")

    # Annotate
    ax_corr.text(
        0.05, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {p_spear:.3f}",
        transform=ax_corr.transAxes,
        ha="left",
        va="top",
        fontsize=9
    )

    sns.despine()
    plt.tight_layout()

    plt.savefig(
        os.path.join(outdir, f"Shannon_vs_age_subject_level_{key}.png"),
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(fig_corr)

In [ ]:
# Plot Shannon Diversity plots for both V1-V3 and V4
for key, biom_path in biom_paths.items():
    calculate_shannon_alpha_diversity_and_plot(
        outdir='../Figures/Supplementary/Suppl_Figure_3/',
        biom_path=biom_path,
        metadata_path=metadata_path,
        group_col='group',
        title_suffix=key,
        keep_samples=shared_samples
    )


[INFO] MixedLM results (V1-V3) — adjusted for: ['age']
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Shannon  
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.1290   
Min. group size:    1        Log-Likelihood:      -100.4900
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         6.036    1.926  3.134 0.002  2.261  9.812
group[T.Acne_NL] -0.639    0.305 -2.096 0.036 -1.237 -0.042
group[T.Acne_L]  -0.489    0.302 -1.623 0.105 -1.081  0.102
age              -0.241    0.091 -2.646 0.008 -0.419 -0.062
Group Var         0.307    0.376                           

           comparison  estimate         z         p
0 

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na


[INFO] Subject-level Shannon vs Age correlation:
Spearman ρ = -0.44
p = 0.0756

[INFO] MixedLM results (V4) — adjusted for: ['age']
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Shannon  
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.4808   
Min. group size:    1        Log-Likelihood:      -219.3810
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept         7.593    3.848  1.973 0.048  0.051 15.135
group[T.Acne_NL] -1.523    0.608 -2.505 0.012 -2.715 -0.332
group[T.Acne_L]  -1.431    0.602 -2.379 0.017 -2.611 -0.252
age              -0.177    0.182 -0.976 0.329 -0.534  0.179
Group Var         1.232    0.736      

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na


[INFO] Subject-level Shannon vs Age correlation:
Spearman ρ = -0.13
p = 0.6135


## Faith Phylogenetic Alpha Diversity

In [155]:
def calculate_faithpd_and_plot(
    biom_path,
    tree_path,
    metadata_path,
    group_col,
    subject_col,
    title_suffix,
    outdir,
    key,
    covariates=('age',)           # NEW: tuple of column names to adjust for
):
    """
    Compute Faith PD alpha diversity, fit a linear mixed-effects model
    adjusting for the given covariates, compute pairwise contrasts, and plot.
    """

    def mixedlm_pairwise_pvalues(df, response, group_col, subject_col, covariates):
        # Build formula: Faith_PD ~ group [+ cov1 + cov2 ...]
        rhs = group_col
        if covariates:
            rhs = rhs + " + " + " + ".join(covariates)
        formula = f"{response} ~ {rhs}"

        model = smf.mixedlm(formula, data=df, groups=df[subject_col])
        result = model.fit()

        params = result.params
        cov = result.cov_params()
        names = params.index.tolist()

        def make_contrast(weights):
            # zeros for every param, including covariates — so group contrasts
            # are automatically "adjusted for age" without extra bookkeeping
            c = np.zeros(len(params))
            for term, w in weights.items():
                c[names.index(term)] = w
            return c

        contrasts = {
            "Healthy vs Acne_NL": make_contrast({f"{group_col}[T.Acne_NL]": 1}),
            "Healthy vs Acne_L":  make_contrast({f"{group_col}[T.Acne_L]": 1}),
            "Acne_NL vs Acne_L":  make_contrast({
                f"{group_col}[T.Acne_L]": 1,
                f"{group_col}[T.Acne_NL]": -1
            })
        }

        rows = []
        for name, c in contrasts.items():
            est = np.dot(c, params.values)
            se = np.sqrt(np.dot(c, np.dot(cov.values, c)))
            z  = est / se
            p  = 2 * (1 - stats.norm.cdf(abs(z)))
            rows.append(dict(comparison=name, p=p))

        return pd.DataFrame(rows), result

    table = load_table(biom_path)
    tree = TreeNode.read(tree_path)

    counts = table.to_dataframe(dense=True).T

    faith = {
        sid: faith_pd(counts.loc[sid].values, counts.columns, tree)
        for sid in counts.index
    }

    faith_df = (
        pd.DataFrame.from_dict(faith, orient="index", columns=["Faith_PD"])
        .reset_index()
        .rename(columns={"index": "SampleID"})
    )

    faith_df["SampleID"] = (
        faith_df["SampleID"].astype(str)
        .str.replace(r"^.*?(?=LAMI)", "", regex=True)
    )

    faith_df.to_csv(
        f"../Analyses/16S/diversity/FaithPD_{key}.csv",
        index=False
    )

    metadata = pd.read_csv(metadata_path, sep="\t")
    metadata["SampleID"] = (
        metadata["SampleID"].astype(str)
        .str.replace(r"^.*?(?=LAMI)", "", regex=True)
    )

    metadata = metadata.merge(faith_df, on="SampleID", how="inner")

    # Coerce covariates to numeric (TSV loads everything as string/object)
    covariates = list(covariates) if covariates else []
    for cov_col in covariates:
        if cov_col in metadata.columns:
            metadata[cov_col] = pd.to_numeric(metadata[cov_col], errors="coerce")
        else:
            raise KeyError(f"Covariate '{cov_col}' not found in metadata columns.")

    # Drop rows missing any model variable (now includes covariates)
    required = [group_col, subject_col, "Faith_PD"] + covariates
    metadata = metadata.dropna(subset=required)

    metadata[group_col] = pd.Categorical(
        metadata[group_col],
        categories=["Healthy", "Acne_NL", "Acne_L"],
        ordered=False
    )

    metadata["severity_category"] = pd.cut(
        metadata["local_lesion_severity"],
        bins=[0, 2, 4, 6],
        labels=["Low", "Mod", "High"],
        include_lowest=True
    )
    
    plot_order = ["Healthy", "Acne_NL", "Acne_L"]
    plot_order = [g for g in plot_order if g in metadata[group_col].unique()]

    pairwise_df, lmm_result = mixedlm_pairwise_pvalues(
        metadata, "Faith_PD", group_col, subject_col, covariates
    )

    print(f"\n[INFO] Faith PD MixedLM results ({key}) — adjusted for: {covariates or 'none'}")
    print(lmm_result.summary())
    print(pairwise_df)

    palette = {
        "Healthy": "#23238f",
        "Acne_NL": "#44a7b6",
        "Acne_L": "#d8543e"
    }

    severity_palette = {
        "Low": "#F1948A",
        "Mod": "#EC7063",
        "High": "#C0392B"
    }

    fig, ax = plt.subplots(figsize=(3.5, 6))

    sns.boxplot(
        data=metadata,
        x=group_col,
        y="Faith_PD",
        order=plot_order,
        palette=palette,
        fliersize=0,
        linewidth=1.2,
        ax=ax
    )

    sns.stripplot(
        data=metadata[metadata[group_col] != "Acne_L"],
        x=group_col,
        y="Faith_PD",
        order=plot_order,
        hue=group_col,
        hue_order=["Healthy", "Acne_NL"],
        palette=palette,
        jitter=True,
        dodge=False,
        size=5,
        edgecolor="black",
        linewidth=0.6,
        ax=ax
    )

    ax.legend_.remove()

    acne_l = metadata[metadata[group_col] == "Acne_L"]
    if not acne_l.empty:
        sns.stripplot(
            data=acne_l,
            x=group_col,
            y="Faith_PD",
            order=plot_order,
            hue="severity_category",
            hue_order=["Low", "Mod", "High"],
            palette=severity_palette,
            jitter=True,
            dodge=False,
            size=5,
            edgecolor="black",
            linewidth=0.6,
            ax=ax
        )

        handles, labels = ax.get_legend_handles_labels()
        handles = handles[-3:]
        labels = ["Low (1–2)", "Mod (3–4)", "High (5–6)"]

        legend_pos = {
            "V1-V3": dict(loc="upper right", bbox_to_anchor=(0.91, 0.87)),
            "V4":    dict(loc="upper left",  bbox_to_anchor=(0.03, 0.2)),
        }

        ax.legend(
            handles,
            labels,
            title="Severity",
            fontsize=8,
            title_fontsize=9,
            frameon=True,
            **legend_pos.get(key, {})
        )

    ymax = metadata["Faith_PD"].max()
    step = (metadata["Faith_PD"].max() - metadata["Faith_PD"].min()) * 0.08
    current_y = ymax + step

    comparison_map = {
        "Healthy vs Acne_NL": ("Healthy", "Acne_NL"),
        "Healthy vs Acne_L":  ("Healthy", "Acne_L"),
        "Acne_NL vs Acne_L":  ("Acne_NL", "Acne_L")
    }

    for _, row in pairwise_df.iterrows():
        g1, g2 = comparison_map[row["comparison"]]
        x1 = plot_order.index(g1)
        x2 = plot_order.index(g2)

        p = row["p"]
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        label = f"{stars} p={p:.4f}".strip()

        ax.plot(
            [x1, x1, x2, x2],
            [current_y, current_y + step * 0.2,
             current_y + step * 0.2, current_y],
            lw=1,
            c="black"
        )

        ax.text(
            (x1 + x2) / 2,
            current_y + step * 0.25,
            label,
            ha="center",
            va="bottom",
            fontsize=9
        )

        current_y += step * 1.1

    label_map = {"Healthy": "H", "Acne_NL": "ANL", "Acne_L": "AL"}
    counts = metadata[group_col].value_counts().to_dict()

    ax.set_xticklabels([
        f"{label_map[g]}\n(n={counts.get(g, 0)})"
        for g in plot_order
    ])

    ax.set_xlabel("")
    ax.set_ylabel("Faith PD")
    ax.set_title(f"16S rRNA ({title_suffix}) Alpha Diversity")

    sns.despine()
    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    plt.savefig(
        os.path.join(outdir, f"FaithPD_alpha_diversity_{key}.png"),
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(fig)

    # Plot diversity correlation with age
    # ---- Aggregate to subject-level ----
    subject_df = (
        metadata
        .groupby(subject_col)
        .agg({
            "Faith_PD": "mean",
            "age": "first"   # assumes age is constant per subject
        })
        .reset_index()
    )

    # ---- Spearman correlation (subject-level) ----
    rho, p_spear = spearmanr(subject_df["Faith_PD"], subject_df["age"])

    print(f"\n[INFO] Subject-level Faith_PD vs Age correlation:")
    print(f"Spearman ρ = {rho:.2f}\np = {p_spear:.4f}")

    # ---- Correlation plot (subject-level) ----
    fig_corr, ax_corr = plt.subplots(figsize=(4, 4))

    sns.regplot(
        data=subject_df,
        x="age",
        y="Faith_PD",
        scatter_kws=dict(s=50, edgecolor="black", linewidth=0.6),
        line_kws=dict(color="black", linewidth=1),
        ax=ax_corr
    )

    ax_corr.set_xlabel("Age")
    ax_corr.set_ylabel("Mean Faith PD per subject")
    ax_corr.set_title(f"Subject-level Faith PD vs Age ({title_suffix})")

    # Annotate
    ax_corr.text(
        0.05, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {p_spear:.3f}",
        transform=ax_corr.transAxes,
        ha="left",
        va="top",
        fontsize=9
    )

    sns.despine()
    plt.tight_layout()

    plt.savefig(
        os.path.join(outdir, f"FaithPD_vs_age_subject_level_{key}.png"),
        dpi=600,
        bbox_inches="tight"
    )
    plt.close(fig_corr)

In [ ]:
# Define inputs
tree_paths = {
    "V1-V3": "../Data/16S/Trees/179426_V1V3_ASVs_seqIDs.rooted.nwk",
    "V4":    "../Data/16S/Trees/174951_V4_ASVs_seqIDs.rooted.nwk"
}

outdir = "../Figures/Main/Figure_3/"


In [157]:
# Run Faith PD for V1-V3
calculate_faithpd_and_plot(
    biom_path=biom_paths["V1-V3"],
    tree_path=tree_paths["V1-V3"],
    metadata_path=metadata_path,
    group_col="group",
    title_suffix="V1–V3",
    outdir=outdir,
    key="V1-V3",
    subject_col="subject_ID"
)



[INFO] Faith PD MixedLM results (V1-V3) — adjusted for: ['age']
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Faith_PD 
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.4317   
Min. group size:    1        Log-Likelihood:      -203.7400
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept        11.691    2.394  4.884 0.000  6.999 16.384
group[T.Acne_NL] -1.395    0.388 -3.592 0.000 -2.156 -0.634
group[T.Acne_L]  -0.995    0.380 -2.622 0.009 -1.739 -0.251
age              -0.348    0.113 -3.080 0.002 -0.570 -0.127
Group Var         0.424    0.292                           

           comparison         p
0  Healthy vs

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na


[INFO] Subject-level Faith_PD vs Age correlation:
Spearman ρ = -0.35
p = 0.1685


In [158]:
# Run Faith PD for V4
calculate_faithpd_and_plot(
    biom_path=biom_paths["V4"],
    tree_path=tree_paths["V4"],
    metadata_path=metadata_path,
    group_col="group",
    title_suffix="V4",
    outdir=outdir,
    key="V4",
    subject_col="subject_ID"
)



[INFO] Faith PD MixedLM results (V4) — adjusted for: ['age']
           Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  Faith_PD 
No. Observations:   184      Method:              REML     
No. Groups:         17       Scale:               0.9439   
Min. group size:    1        Log-Likelihood:      -282.2485
Max. group size:    20       Converged:           Yes      
Mean group size:    10.8                                   
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept        17.994    6.277  2.867 0.004  5.692 30.297
group[T.Acne_NL] -1.917    0.987 -1.943 0.052 -3.852  0.017
group[T.Acne_L]  -1.661    0.979 -1.696 0.090 -3.581  0.259
age              -0.387    0.297 -1.306 0.192 -0.969  0.194
Group Var         3.362    1.394                           

           comparison         p
0  Healthy vs Ac

/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na


[INFO] Subject-level Faith_PD vs Age correlation:
Spearman ρ = -0.20
p = 0.4316
